In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
import os
import requests
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
load_dotenv()

True

In [10]:
huggingfacehub_api_token = os.getenv('HUGGINGFACEHUB_ACCESS_TOKEN')
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    huggingfacehub_api_token=huggingfacehub_api_token,
)
model = ChatHuggingFace(llm=llm)
tool = DuckDuckGoSearchRun()

In [11]:
# Agent imports
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

# Step 2: Pull the ReAct (Reasoning + Action) prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [12]:
agent = create_react_agent(
    llm = model,
    tools = [tool],
    prompt = prompt
) # agent who plans task and all

In [13]:
# Now we need to wrap it with AgenExecutor (It is the one that executes what the agent says)
agent_executor = AgentExecutor(
    agent = agent,
    tools = [tool],
    # max_iterations=3, # limits the agent to at most 3 thought–action loops before stopping
    # early_stopping_method="generate", # if the agent hits the iteration limit, force the LLM to generate a final answer from the gathered observations
    verbose = True
)

In [14]:
response = agent_executor.invoke({
    "input": input("Enter your query: ")
})['output']
print(response)



> Entering new AgentExecutor chain...
Thought: I need to determine the financial capital of India and then find its population.
Action: duckduckgo_search
Action Input: financial capital of India
ObservMumbai is thefinancial, commercial, [30] and entertainment capital ofIndia. Mumbai is often compared to New York City, [31][32] and is home to the Bombay Stock Exchange, situated on Dalal Street. May 1, 2025 ·Mumbai, formerly known as Bombay, isIndia’s most populous city and its financial powerhouse. Located on the western coast, it’s home to over 20 million people and serves as the capital of Maharashtra. Nov 5, 2025 ·Let’s find out. Mumbai has always been known as thefinancialcapitalofIndia. The city acts as the powerhouse that drives the economy of the nation. The capital city is also known as the ‘City of Dreams,’ as it boasts one of the biggest business, trade, banking, and entertainment centres in India. Mumbai, thefinancialcapitalofIndia, has the highest number of billionaires in